# Suite BRG — The Python bridge

`Bridges.Python` carries expressions between mbse-expressions' Python dialect and the Python language's syntax
trees, both ways, tree to tree. Each case writes terms as syntax nodes, whose printed Python is the dialect's own
render, and reads syntax nodes as terms: with the Python standards' parsers, this reads Python source into the
dialect, which the dialect alone does only in Python. Python's suite also checks the bridge against the dialect's own
`parse`, against Python's reading of literals, and runs the functions it writes.

In [ ]:
import { Evaluators, Expressions as E, Text } from "@mbse/expressions/Dialects/Python";
import { Terms as F } from "@mbse/expressions/Framework";
import { Python as B } from "@mbse/programs/Bridges";
import { Errors } from "@mbse/programs/Framework";
import { Python314, Syntax as P } from "@mbse/programs/Python";
import { Evaluators as BasicEvaluators, Expressions as Basic } from "@mbse/expressions";
import * as Translators from "@mbse/expressions/Translators";
import { Proxies, Schemas as S } from "@mbse/schemas/Framework";

import { assert, equal, raises } from "./support.js";

const Py = P.LANGUAGE.Builders;
const printed = (term: any) => Python314.print(B.module(term)).replace(/\n+$/, "");
const read = (source: string) => B.term_of_module(Python314.parse(source) as P.Module);

function fails(source: string, message: string): void {
  raises(Errors.TranspileError, () => read(source), message);
}

## BRG-01 · Every term of the dialect writes the Python the dialect renders, and reads back the same

In [ ]:
const [x, y, z] = [E.name("x"), E.name("y"), E.name("z")];
const TERMS: any[] = [
  E.constant(1n), E.name("this"), E.attribute(x, "age"), E.subscript(x, "key"), E.index(x, E.name("i")),
  E.call("np.add", 1n, 2n), E.compare("not in", x, E.name("s")), E.boolop("and", E.boolop("and", x, y), z),
  E.boolop("or", E.boolop("and", x, y), E.boolop("or", z, E.name("w"))), E.binop("**", E.unaryop("-", x), 2n),
  E.binop("-", x, E.binop("-", y, z)), E.unaryop("not", E.compare("==", x, 1n)), E.ifexp(x, 1n, 2n),
  E.ifexp(E.ifexp(x, 1n, 2n), 3n, 4n), E.call("all", E.generator("p", E.name("ports"), E.name("p"))),
  E.generator("p", E.name("ports"), E.attribute(E.name("p"), "pin"), E.compare(">", E.name("p"), 0n), x),
  E.let_("n", 3n, E.binop("*", E.name("n"), E.name("n"))),
  E.import_("numpy", E.call("np.greater_equal", x, 18n), "np"),
  E.import_("os.path", E.call("os.path.join", "a", "b")),
  E.importfrom("math", "sqrt", E.importfrom("math", "pi", E.call("sqrt", E.name("PI")), "PI")),
];
for (const term of TERMS) {
  equal(printed(term), Text.ToText(term));
  assert(F.same(read(printed(term)), term), printed(term));
}
equal(Python314.print(B.expression(TERMS[5])), "np.add(1, 2)");
const enclosing = "an import can only enclose the whole expression";
raises(Errors.TranspileError, () => B.expression(TERMS[TERMS.length - 1]), enclosing);
raises(Errors.TranspileError, () => B.module(E.call("f", E.import_("os", x))), enclosing);
console.log(TERMS.length, "terms");

## BRG-02 · Constants: a native value is the literal `repr` writes, a negative number a negation, and back

In [ ]:
const CONSTANTS: [any, any][] = [
  [true, "True"],
  [false, "False"],
  [0n, "0"],
  [-1n, "-1"],
  [1180591620717411303424n, "1180591620717411303424"],
  [1.5, "1.5"],
  [-0, "-0.0"],
  [1e+16, "1e+16"],
  [1e-05, "1e-05"],
  [Infinity, "float('inf')"],
  [-Infinity, "-float('inf')"],
  [NaN, "float('nan')"],
  ["a'b\"c\n", "'a\\'b\"c\\n'"],
  ["é😀", "'é😀'"],
  [new Uint8Array([0, 255]), "b'\\x00\\xff'"],
  ["", "''"],
];
for (const [value, text] of CONSTANTS) {
  const term = E.constant(value);
  equal(printed(term), text);
  equal(Text.ToText(term), text);
  equal(Text.ToText(read(text)), text);
}
console.log("ok");

## BRG-03 · Python source reads into terms as the dialect parses it: parentheses, chains, lets, generators, imports

In [ ]:
const SOURCES: [any, any][] = [
  ["this.age >= 18 and this.email != ''", "this.age >= 18 and this.email != ''"],
  ["a and b and c or d", "a and b and c or d"],
  ["x['key'] + x[0] - x[i]", "x['key'] + x[0] - x[i]"],
  ["all(p.pin > 0 for p in ports if p.used)", "all(p.pin > 0 for p in ports if p.used)"],
  ["(lambda n: n * n)(3)", "(lambda n: n * n)(3)"],
  ["import numpy as np\nnp.greater_equal(x, 18)", "import numpy as np\nnp.greater_equal(x, 18)"],
  ["import os, json\njson.dumps(os.sep)", "import os\nimport json\njson.dumps(os.sep)"],
  ["from math import sqrt, pi as PI\nsqrt(PI)", "from math import sqrt\nfrom math import pi as PI\nsqrt(PI)"],
  ["-1 + -2.5 - (-x) ** 2", "-1 + -2.5 - (-x) ** 2"],
  ["not a if b else ~c", "not a if b else ~c"],
  ["'a' 'b' + b'c' b'd'", "'ab' + b'cd'"],
  ["f(-0.0, 1e16, 1e-05, 2 ** 70, float('nan'))", "f(-0.0, 1e+16, 1e-05, 2 ** 70, float('nan'))"],
  ["x // y % z << 1 | 3 & 4 ^ 5", "x // y % z << 1 | 3 & 4 ^ 5"],
  ["a in b and a not in c", "a in b and a not in c"],
  ["# a comment\nimport json\njson.dumps(v)  # and another", "import json\njson.dumps(v)"],
  ["((a))[('k')]", "a['k']"],
  ["a < b <= c", "a < b and b <= c"],
];
for (const [source, rendered] of SOURCES) {
  const term = read(source);
  equal(Text.ToText(term), rendered);
  assert(F.same(read(printed(term)), term), printed(term));
}
equal(Text.ToText(B.term((Python314.parse("x + 1") as P.Module).body[0]!.get("value") as any)), "x + 1");
const chain = read("a < b <= c") as any; // its middle operand is one term, shared, as the dialect's FromText makes it
assert(chain.left.right === chain.right.left);
console.log("ok");

## BRG-04 · Literals are decoded as Python decodes them: numbers, prefixes, quotes and escapes

In [ ]:
const LITERALS: [any, any][] = [
  ["True", true],
  ["0x_1F", 31n],
  ["0o17", 15n],
  ["0b1_0", 2n],
  ["1_000", 1000n],
  ["1.5e-3", 0.0015],
  ["1.", 1.0],
  [".5", 0.5],
  ["1_0.5", 10.5],
  ["'a\\'b'", "a'b"],
  ["\"\\x41\\101\\u00e9\\U0001F600\"", "AAé😀"],
  ["'\\a\\b\\f\\n\\r\\t\\v\\\\\\\"'", "\u0007\b\f\n\r\t\u000b\\\""],
  ["'\\q\\8'", "\\q\\8"],
  ["'\\777'", "ǿ"],
  ["'\\0'", "\u0000"],
  ["'''tri\\\nple'''", "triple"],
  ["\"\"\"x\"\"\"", "x"],
  ["r'\\n'", "\\n"],
  ["U'x'", "x"],
  ["b'\\777\\x7f'", new Uint8Array([255, 127])],
  ["b'\\0\\u'", new Uint8Array([0, 92, 117])],
  ["rb'\\x'", new Uint8Array([92, 120])],
  ["Rb'x'", new Uint8Array([120])],
];
for (const [spelling, value] of LITERALS) equal(B.decode(spelling), value);
console.log("ok");

## BRG-05 · A rule becomes a function, its imports first, that returns what the dialect evaluates

In [ ]:
const rule = E.boolop("and", E.compare(">=", E.name("age"), 18n), E.compare(">", E.call("len", E.name("email")), 0n));
const contactable = B.function_("is_contactable", ["age", "email"], rule);
equal(Python314.print(contactable), "def is_contactable(age, email):\n    return age >= 18 and len(email) > 0");
const PEOPLE: [any, any][] = [
  [20n, "a@b"],
  [17n, "a@b"],
  [30n, ""],
];
equal(PEOPLE.map(([age, email]) => Evaluators.OfAny(rule, { age, email })), [true, false, false]);
const distance = E.importfrom("math", "sqrt", E.call("sqrt", E.binop("+", E.binop("**", x, 2n), E.binop("**", y, 2n))));
equal(Python314.print(B.function_("distance", ["x", "y"], distance)),
  "def distance(x, y):\n    from math import sqrt\n    return sqrt(x ** 2 + y ** 2)");
console.log("ok");

## BRG-06 · What the dialect cannot hold fails, at the syntax node's path

In [ ]:
const ERRORS: [any, any][] = [
  ["a is not None", "body[0].value.comparisons[0]: the operator 'is not' has no counterpart in the Python dialect"],
  ["a @ b", "body[0].value: the operator '@' has no counterpart in the Python dialect"],
  ["f(x=1)", "body[0].value.keywords[0]: keyword arguments are not supported"],
  ["x[1:2]", "body[0].value.slice: slices are not supported"],
  ["x[1, 2]", "body[0].value.slice: slices are not supported"],
  ["(lambda a, b: a)(1)", "body[0].value: a let binds one name"],
  ["(lambda a: a)(1, 2)", "body[0].value: a let binds one name"],
  ["(lambda p, /, a: a)(1)", "body[0].value: a let binds one name"],
  ["(lambda a, *r: a)(1)", "body[0].value: a let binds one name"],
  ["(lambda a, *, k: a)(1)", "body[0].value: a let binds one name"],
  ["(lambda a, **k: a)(1)", "body[0].value: a let binds one name"],
  ["(lambda a=1: a)(2)", "body[0].value: a let binds one name"],
  ["(x for x in a for y in b)", "body[0].value: a generator has one for, over a name"],
  ["(x async for x in a)", "body[0].value: a generator has one for, over a name"],
  ["(x for x, y in a)", "body[0].value: a generator has one for, over a name"],
  ["None", "body[0].value: only native constants are supported"],
  ["...", "body[0].value: only native constants are supported"],
  ["1j", "body[0].value: only native constants are supported"],
  ["'\\N{BULLET}'", "body[0].value: a \\N{...} escape is not supported"],
  ["'\\U00110000'", "body[0].value: an escape beyond U+10FFFF"],
  ["f'{x}'", "body[0].value: JoinedStr has no counterpart in the Python dialect"],
  ["'a' f'{x}'", "body[0].value.values[1]: JoinedStr has no counterpart in the Python dialect"],
  ["'a' b'b'", "body[0].value: bytes and strings cannot be joined"],
  ["[1, 2]", "body[0].value: List has no counterpart in the Python dialect"],
  ["x = 1", "the module must end with an expression"],
  ["import a", "the module must end with an expression"],
  ["import a\nb = 1\nc", "body[1]: only imports may precede the expression"],
  ["from . import a\na", "body[0]: only imports may precede the expression"],
  ["from m import *\na", "body[0]: import * binds names that cannot be known"],
];
for (const [source, message] of ERRORS) fails(source, message);
for (const statement of [Py.Import().is_lazy(true).add_names(Py.Alias().name(Py.DottedName().add_names("a"))),
  Py.ImportFrom().is_lazy(true).module(Py.DottedName().add_names("m"))
    .add_names(Py.Alias().name(Py.DottedName().add_names("a")))]) {
  const lazy = Py.Module().body([statement, Py.Expr().value(Py.Name().id("a"))]).create() as P.Module;
  // a lazy import
  raises(Errors.TranspileError, () => B.term_of_module(lazy), "body[0]: only imports may precede the expression");
}
console.log("ok");

## BRG-07 · Rules round-trip through the translators, the bridge, the printer and the parser, and agree with Basic where it decides

In [ ]:
const Shipment = new S.OfObject.Builder().name("Shipment").ref().properties(
  (p: any) => p.name("celsius").of((t: any) => t.as_native(Number)), (p: any) => p.name("hours").of((t: any) => t.as_native(BigInt)),
  (p: any) => p.name("readings").of((t: any) => t.as_indexed((i: any) => i.of((n: any) => n.as_native(Number))))).create();
const store: any = new Proxies.OfStore();
store.register(Shipment);
const shipments = [store.Shipment().celsius(5.5).hours(30n).readings([4.0, 8.6, 9.1]).create(),
  store.Shipment().celsius(9.1).hours(80n).readings([4.0, 12.5]).create(),
  store.Shipment().hours(20n).readings([]).create()]; // no temperature, no readings
const [self, c] = [Basic.variable("this"), Basic.variable("c")];
const RULES = [
  Basic.literal(2.0).le(self.celsius).and_(self.celsius.le(8.0)), self.has("celsius"),
  self.hours.gt(72n).implies(self.celsius.le(8.0)), self.hours.gt(72n).not_(),
  Basic.let_("t", self.celsius, Basic.variable("t").ge(2.0)), self.readings.all("c", c.le(12.0)),
  self.readings.any("c", c.gt(8.0)), self.readings.count_where("c", c.gt(8.0)).le(2n), self.readings.count().gt(0n),
  self.readings.max().le(12.0), self.readings.item(0n).ge(2.0), Basic.literal(4.0).in_(self.readings),
  self.readings.sum().le(30.0), self.hours.add(10n).mul(2n).sub(1n).gt(100n),
];
const toPython = Translators.between(Basic.DIALECT, E.DIALECT);
const toBasic = Translators.between(E.DIALECT, Basic.DIALECT);
for (const rule of RULES) {
  const term = toPython.forward(rule.data);
  const source = Python314.print(B.function_("check", ["this"], term));
  const fn = (Python314.parse(source) as P.Module).body[0] as P.FunctionDef;
  const readBack = B.term((fn.body[0] as P.Return).value as any);
  assert(F.same(readBack, term) && F.same(toBasic.forward(readBack), rule.data), source);
  for (const shipment of shipments) { // where Basic decides, Python's rules agree
    const decided = BasicEvaluators.predicate(rule, shipment);
    if (decided !== null) assert(Evaluators.OfAny(readBack, { this: shipment }) === decided, `${source} ${decided}`);
  }
}
// Where Basic does not, Python's rules decide their own way: an empty sum is the int 0, which Python compares with 30.0.
assert(BasicEvaluators.predicate(RULES[12], shipments[2]) === null);
assert(Evaluators.OfAny(toPython.forward(RULES[12]!.data), { this: shipments[2] }) === true);
console.log(RULES.length, "rules");
